In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 8
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.977866271155252

Trial 1 =========================================
15.879831011084438



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 2 =========================================
16.172256719384983

Trial 3 =========================================
14.337236567100138

Trial 4 =========================================
16.473117872769734

Trial 5 =========================================
16.223216562830814

Trial 6 =========================================
14.473191461241154

Trial 7 =========================================
13.715365627892073

Trial 8 =========================================
18.761702219022965

Trial 9 =========================================
14.141140513124324

Trial 10 =========================================
13.449759483591492

Trial 11 =========================================
15.112783829189159

Trial 12 =========================================
16.03461187386605

Trial 13 =========================================
14.731759747859927

Trial 14 =========================================
14.639073627601347

Trial 15 =========================================
15.227376038768293

Trial 16 ======

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 32 =========================================
16.23630088279135

Trial 33 =========================================
15.338751443285586

Trial 34 =========================================
15.98892905569816

Trial 35 =========================================
16.49619703611249

Trial 36 =========================================
16.20318065431936

Trial 37 =========================================
16.785081830756653

Trial 38 =========================================
15.270578914480486

Trial 39 =========================================
14.364990806391583

Trial 40 =========================================
15.351901228963891

Trial 41 =========================================
16.205772718811442

Trial 42 =========================================
14.945657399757

Trial 43 =========================================
14.146183865785648

Trial 44 =========================================
14.606747620612598

Trial 45 =========================================
15.07821933276986

Trial 46 =====

...

Trial 64 =========================================
16.234845610706067

Trial 65 =========================================
16.24255477239929

Trial 66 =========================================
15.989650992285274



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/mi

Trial 67 =========================================
16.504229527055717

Trial 68 =========================================
14.196806972730679

Trial 69 =========================================
14.840584370702798

Trial 70 =========================================
15.394119224840196

Trial 71 =========================================
18.8105764745952

Trial 72 =========================================
14.29540022857996

Trial 73 =========================================
14.450916258240206

Trial 74 =========================================
14.28505964349299

Trial 75 =========================================
15.395654910910423

Trial 76 =========================================
17.48369531927088

Trial 77 =========================================
15.945294240975663

Trial 78 =========================================
17.416042938661924

Trial 79 =========================================
15.39081184077228

Trial 80 =========================================
16.215016727597074

Trial 81 ===

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 93 =========================================
15.533942467350524

Trial 94 =========================================
18.798601871576583

Trial 95 =========================================
15.39258229576084

Trial 96 =========================================
15.095169453451375

Trial 97 =========================================
15.37573391566623

Trial 98 =========================================
14.627954453777424

Trial 99 =========================================
14.213698044938713

[13.97786627 15.87983101 16.17225672 14.33723657 16.47311787 16.22321656
 14.47319146 13.71536563 18.76170222 14.14114051 13.44975948 15.11278383
 16.03461187 14.73175975 14.63907363 15.22737604 16.17973307 13.7869619
 15.39095806 14.86143508 14.77813034 15.39685956 14.05936852 14.15584559
 15.39768055 14.16226719 16.01890727 16.24190446 18.77443769 14.44510665
 16.21089966 14.63209127 16.23630088 15.33875144 15.98892906 16.49619704
 16.20318065 16.78508183 15.27057891 14.36499081 15.35190123 16.2057

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.816099677525994
Avg = 15.482505334274526
Std = 1.2609957108199978


In [6]:
print(y_max_arr.tolist())

[13.977866271155252, 15.879831011084438, 16.172256719384983, 14.337236567100138, 16.473117872769734, 16.223216562830814, 14.473191461241154, 13.715365627892073, 18.761702219022965, 14.141140513124324, 13.449759483591492, 15.112783829189159, 16.03461187386605, 14.731759747859927, 14.639073627601347, 15.227376038768293, 16.179733065585733, 13.786961898493507, 15.390958057416743, 14.86143507663781, 14.778130340375068, 15.396859560210215, 14.059368517898479, 14.155845593048832, 15.397680546310978, 14.162267194623023, 16.018907273285098, 16.241904463535995, 18.77443769214309, 14.44510664610605, 16.210899660262218, 14.632091271125818, 16.23630088279135, 15.338751443285586, 15.98892905569816, 16.49619703611249, 16.20318065431936, 16.785081830756653, 15.270578914480486, 14.364990806391583, 15.351901228963891, 16.205772718811442, 14.945657399757, 14.146183865785648, 14.606747620612598, 15.07821933276986, 14.296068411139421, 15.110581720591625, 14.61104132264864, 15.61741332663615, 16.2330077094

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-8.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)